# Training - split y feature engineering

Este notebook inicia la etapa de modelado de churn usando el flujo de `ds-feature` en modo ML: split primero, transformaciones despues, sin entrenamiento de modelos ni feature selection.

**Input de trabajo:** `data/processed/datos_limpios.csv`, generado por el notebook de limpieza. Las categorias equivalentes ya vienen estandarizadas desde esa etapa.

**Contrato del notebook:** generar `data/processed/split/`, `features_train.parquet`, `features_test.parquet` y dejar el pipeline reproducible en `src/features/pipeline.py`.

## 0. Setup

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.features.pipeline import (
    CATEGORICAL_FEATURES,
    DROP_COLUMNS,
    ID_COL,
    NUMERIC_FEATURES,
    TARGET_COL,
    build_pipeline,
    transform_to_dataframe,
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "datos_limpios.csv"
SPLIT_DIR = PROJECT_ROOT / "data" / "processed" / "split"
FEATURES_TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "features_train.parquet"
FEATURES_TEST_PATH = PROJECT_ROOT / "data" / "processed" / "features_test.parquet"
TARGET_TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "target_train.csv"
TARGET_TEST_PATH = PROJECT_ROOT / "data" / "processed" / "target_test.csv"

## 1. Carga y validaciones minimas

Validamos lo minimo antes del split: columnas requeridas, target sin nulos, IDs unicos y duplicados completos.

In [2]:
df = pd.read_csv(DATA_PATH)

required_cols = {ID_COL, TARGET_COL}
missing_required = required_cols - set(df.columns)
if missing_required:
    raise ValueError(f"Faltan columnas requeridas: {sorted(missing_required)}")

if df[ID_COL].duplicated().any():
    raise ValueError(f"La columna {ID_COL} tiene IDs duplicados.")

if df[TARGET_COL].isna().any():
    raise ValueError(f"La columna target {TARGET_COL} contiene nulos.")

print(f"Shape input limpio: {df.shape}")
print(f"Filas duplicadas completas: {df.duplicated().sum()}")
print(f"Nulos totales raw: {df.isna().sum().sum()}")

Shape input limpio: (5630, 20)
Filas duplicadas completas: 0
Nulos totales raw: 0


## 2. Target y split estratificado

`Churn = 1` representa abandono. Como la clase positiva ronda el 17%, usamos `stratify=y` para mantener la proporcion de churners en train y test.

In [3]:
target_distribution = df[TARGET_COL].value_counts().sort_index().rename_axis(TARGET_COL).reset_index(name="n")
target_distribution["pct"] = target_distribution["n"] / len(df)
target_distribution

,Churn,n,pct
0,0,4682,0.831616
1,1,948,0.168384


In [4]:
y = df[TARGET_COL].copy()

train_idx, test_idx = train_test_split(
    df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

train_df = df.loc[train_idx].sort_index().copy()
test_df = df.loc[test_idx].sort_index().copy()

X_train = train_df.drop(columns=DROP_COLUMNS).copy()
X_test = test_df.drop(columns=DROP_COLUMNS).copy()
y_train = train_df[[ID_COL, TARGET_COL]].copy()
y_test = test_df[[ID_COL, TARGET_COL]].copy()

print(f"Train: {len(train_df)} filas")
print(f"Test:  {len(test_df)} filas")
print(f"Features raw candidatas despues de drops: {X_train.shape[1]}")

Train: 4504 filas
Test:  1126 filas
Features raw candidatas despues de drops: 17


## 3. Verificacion del split

In [5]:
def class_summary(y_series: pd.Series, split_name: str) -> pd.DataFrame:
    summary = y_series.value_counts().sort_index().rename_axis(TARGET_COL).reset_index(name="n")
    summary["pct"] = summary["n"] / len(y_series)
    summary.insert(0, "split", split_name)
    return summary

split_summary = pd.concat(
    [
        class_summary(df[TARGET_COL], "full"),
        class_summary(train_df[TARGET_COL], "train"),
        class_summary(test_df[TARGET_COL], "test"),
    ],
    ignore_index=True,
)

assert set(train_idx).isdisjoint(set(test_idx)), "Hay solapamiento entre train y test."
assert len(train_idx) + len(test_idx) == len(df), "El split no cubre todas las filas."

split_summary

,split,Churn,n,pct
0,full,0,4682,0.831616
1,full,1,948,0.168384
2,train,0,3746,0.831705
3,train,1,758,0.168295
4,test,0,936,0.831261
5,test,1,190,0.168739


## 4. Guardado de CSVs de split

La carpeta `data/processed/split/` guarda los CSVs separados de train/test y los indices para reconstruir el holdout. Estos archivos son datos sin transformar; el pipeline se fitea despues solo con train.

In [6]:
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

train_indices = train_df[[ID_COL]].copy()
train_indices.insert(0, "row_index", train_df.index)

test_indices = test_df[[ID_COL]].copy()
test_indices.insert(0, "row_index", test_df.index)

split_indices = pd.concat(
    [train_indices.assign(split="train"), test_indices.assign(split="test")],
    ignore_index=True,
).sort_values("row_index")

train_df.to_csv(SPLIT_DIR / "train.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test.csv", index=False)
X_train.to_csv(SPLIT_DIR / "X_train.csv", index=False)
X_test.to_csv(SPLIT_DIR / "X_test.csv", index=False)
y_train.to_csv(SPLIT_DIR / "y_train.csv", index=False)
y_test.to_csv(SPLIT_DIR / "y_test.csv", index=False)
train_indices.to_csv(SPLIT_DIR / "train_indices.csv", index=False)
test_indices.to_csv(SPLIT_DIR / "test_indices.csv", index=False)
split_indices.to_csv(SPLIT_DIR / "split_indices.csv", index=False)
split_summary.to_csv(SPLIT_DIR / "split_summary.csv", index=False)

pd.DataFrame(
    {
        "archivo": sorted(path.name for path in SPLIT_DIR.glob("*.csv")),
    }
)

,archivo
0,X_test.csv
1,X_train.csv
2,split_indices.csv
3,split_summary.csv
4,test.csv
5,test_indices.csv
6,train.csv
7,train_indices.csv
8,y_test.csv
9,y_train.csv


## 5. Pipeline de features

El pipeline vive en `src/features/pipeline.py` y aplica:

- drop de `CustomerID`, `Churn` y `DaySinceLastOrder`;
- features derivadas de negocio respaldadas por el EDA: proxy de valor, ratios por orden e interaccion reclamo-satisfaccion;
- imputacion numerica con mediana y escalado robusto;
- imputacion categorica con moda y One-Hot Encoding.

Importante: el test entra solo por `.transform()`, nunca por `.fit()`.

In [7]:
feature_pipeline = build_pipeline()
feature_pipeline.fit(X_train, train_df[TARGET_COL])

X_train_features = transform_to_dataframe(feature_pipeline, X_train, index=train_df.index)
X_test_features = transform_to_dataframe(feature_pipeline, X_test, index=test_df.index)

print(f"Features transformadas train: {X_train_features.shape}")
print(f"Features transformadas test:  {X_test_features.shape}")
print(f"Columnas numericas base + derivadas: {len(NUMERIC_FEATURES)}")
print(f"Columnas categoricas OHE: {len(CATEGORICAL_FEATURES)}")

Features transformadas train: (4504, 38)
Features transformadas test:  (1126, 38)
Columnas numericas base + derivadas: 21
Columnas categoricas OHE: 5


## 6. Validaciones post-transformacion

In [8]:
assert X_train_features.shape[0] == len(train_df)
assert X_test_features.shape[0] == len(test_df)
assert X_train_features.shape[1] == X_test_features.shape[1]
assert X_train_features.isna().sum().sum() == 0
assert X_test_features.isna().sum().sum() == 0

validation_summary = pd.DataFrame(
    {
        "check": [
            "split antes de transformar",
            "fit solo en train",
            "test solo transform",
            "sin nulos post-transformacion",
            "mismas columnas train/test",
        ],
        "status": ["OK", "OK", "OK", "OK", "OK"],
    }
)
validation_summary

,check,status
0,split antes de transformar,OK
1,fit solo en train,OK
2,test solo transform,OK
3,sin nulos post-transformacion,OK
4,mismas columnas train/test,OK


## 7. Guardado de features transformadas

Guardamos las features en parquet para el Modeler y los targets por separado. El test queda reservado para evaluacion final.

In [9]:
X_train_features.to_parquet(FEATURES_TRAIN_PATH, index=True)
X_test_features.to_parquet(FEATURES_TEST_PATH, index=True)
y_train.to_csv(TARGET_TRAIN_PATH, index=False)
y_test.to_csv(TARGET_TEST_PATH, index=False)

print(f"Features train: {FEATURES_TRAIN_PATH}")
print(f"Features test:  {FEATURES_TEST_PATH}")
print(f"Target train:   {TARGET_TRAIN_PATH}")
print(f"Target test:    {TARGET_TEST_PATH}")

Features train: /home/berni/Desktop/Dev/TP3_Prediccion_Churn_IAAN_Grupo_04/data/processed/features_train.parquet
Features test:  /home/berni/Desktop/Dev/TP3_Prediccion_Churn_IAAN_Grupo_04/data/processed/features_test.parquet
Target train:   /home/berni/Desktop/Dev/TP3_Prediccion_Churn_IAAN_Grupo_04/data/processed/target_train.csv
Target test:    /home/berni/Desktop/Dev/TP3_Prediccion_Churn_IAAN_Grupo_04/data/processed/target_test.csv


## Proximo paso

El siguiente notebook/bloque ya corresponde a `ds-model`: baseline con `DummyClassifier`, CV estratificada, comparacion de modelos, seleccion de features y tuning de umbral sin tocar el test hasta el final.